# When ASR confidence does not travel: reproduce the results

This notebook reproduces the study in [https://github.com/neric-joel/DL-Project-](https://github.com/neric-joel/DL-Project-) on Google Colab.

* **Part A (about 2 minutes, CPU is fine):** rebuild every table and figure from the per-unit results
  committed in the repository.
* **Part B (about 1 hour on a T4 GPU):** run the whole pipeline from the raw audio: download both
  datasets, run Whisper-small on 11,213 units, score, evaluate, and draw the figures.

For Part B choose *Runtime → Change runtime type → T4 GPU* first.

In [ ]:
!git clone -q https://github.com/neric-joel/DL-Project- asrshift
%cd asrshift
!pip install -q -e .

## Part A: tables and figures from the committed per-unit results

In [ ]:
!asrshift evaluate --n-boot 500
!asrshift figures
!asrshift report

In [ ]:
import pandas as pd
from IPython.display import Image, display

t = "results/tables/small"
asr = pd.read_csv(f"{t}/asr_summary.csv").set_index("set")
display(asr.loc[["eka_test", "pm_turn_test", "pm_window_test"],
                ["units", "hours", "wer", "err_rate", "number_acc", "negation_acc", "mean_conf"]].round(3))
cal = pd.read_csv(f"{t}/err/calibration.csv")
display(cal[cal.policy.isin(["P3", "P4"])][["eval", "policy", "base_rate", "mean_p", "ece", "brier", "citl"]].round(3))
ops = pd.read_csv(f"{t}/err/operating_points.csv")
display(ops[(ops.point == "budget_0.10") & ops.policy.isin(["P1", "P3", "P1-retuned", "P4"])]
        [["eval", "policy", "review_rate", "err_caught", "accepted_err_rate"]].round(3))

In [ ]:
for f in ["fig1_confidence_vs_error", "fig2_reliability", "fig3_risk_coverage", "fig4_threshold_transfer",
          "fig5_recalibration_data"]:
    display(Image(f"results/figures/{f}.png", width=900))

## Part B: the full pipeline from raw audio (T4 GPU)

Downloads about 2.1 GB (Eka parquet shards from the Hugging Face Hub, PriMock57 audio from GitHub
LFS), builds the speaker-disjoint and consultation-level splits, runs faster-whisper, and rewrites
everything under `results/`. Each step is resumable if the runtime disconnects.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!asrshift download
!asrshift prepare

In [ ]:
# ~50 minutes on a T4. Add `--limit 200` for a quick end-to-end smoke test.
!asrshift infer

In [ ]:
!asrshift score
!asrshift evaluate
!asrshift figures
!asrshift report

## One transcript, end to end

In [ ]:
import pandas as pd
from asrshift import audio, infer, prepare, review
from asrshift.config import load_config

cfg = load_config()
m = prepare.load()
row = m[(m.subset == "window") & (m.split == "test")].iloc[40].to_dict()
model, device, ctype = infer.load_model(cfg["asr"])
rec = infer.transcribe(model, audio.load_unit_audio(row), cfg["asr"]["decode"])
print("REFERENCE :", row["reference"][:400])
print("WHISPER   :", rec["text"][:400])
print(review.assess(rec, domain="window"))